In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' can be imported
import sys
from pathlib import Path # for path manipulations
# Move three levels up from current working directory to reach workspace root
workspace_root = Path.cwd().parent.parent.parent.resolve() 
smx_dir = workspace_root / 'smx'  # Path to smx folder
if str(smx_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(smx_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{workspace_root}/real_datasets/xrf/milk.csv', sep=';') # local copy of Toledo 2022 dataset
data = data_complete.loc[:, '2.66':'22.62']

# Creating a new column 'Class' based on the condition of the samples in the 'Type' column being 'Authentic'
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '2.66':'22.62'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '2.66':'22.62'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 16:34:54,819 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-26 16:34:54,842 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [2]:
from modeling import mlp_optimized

mlp_model = mlp_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, 
                          aim='classification', 
                          hidden_layer_sizes=(64,32),
                          activation='tanh',
                          learning_rate='adaptive',
                          max_iter=10,
                          random_state=1)
mlp_model[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,MLP,0.981343,0.970238,1.0,"[[100, 0], [5, 163]]",0.721739,0.625,0.883721,"[[38, 5], [27, 45]]"


## Definição das Zonas Espectrais

In [3]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Ag La', 2.66, 3.10),
('Ag Lb', 3.10, 3.46),
('Ca', 3.46, 3.92),
('background', 3.92, 6.12),
('Fe', 6.12, 6.68),
('Cu', 6.70, 8.37),
('Zn', 8.37, 9.10),
('Bremsstrahlung', 9.10, 20.06),
('Ag compton', 20.06, 21.62),
('Ag ka', 21.48, 22.62)
]

## SHAP

In [4]:
shap_global_importance = pd.read_csv('shap_milk.csv', sep=';') # loading previously saved shap_unique_df
# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df = shap_unique_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,3.17,0.021872,Ag Lb
1,3.02,0.019238,Ag La
2,22.28,0.016172,Ag ka
3,8.73,0.006672,Zn
4,21.18,0.004494,Ag compton
5,16.55,0.003386,Bremsstrahlung
6,6.09,0.001753,background
7,6.86,0.001411,Cu
8,6.40,0.001392,Fe
9,3.58,0.000174,Ca


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [5]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zone 'Ag La': VE = 84.62%, dim = 18 variables
Zone 'Ag Lb': VE = 86.51%, dim = 15 variables
Zone 'Ca': VE = 21.64%, dim = 19 variables
Zone 'background': VE = 3.44%, dim = 87 variables
Zone 'Fe': VE = 12.88%, dim = 23 variables
Zone 'Cu': VE = 9.26%, dim = 66 variables
Zone 'Zn': VE = 17.17%, dim = 29 variables
Zone 'Bremsstrahlung': VE = 30.64%, dim = 429 variables
Zone 'Ag compton': VE = 52.31%, dim = 62 variables
Zone 'Ag ka': VE = 76.93%, dim = 45 variables

Scores DataFrame shape: (268, 10)


In [6]:
y_pred = mlp_model[4]['MLP'].values # using the continuous predictions from SVM, extracting as 1D array
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred
)

In [7]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_pred = pd.Series(mlp_model[4]['MLP'].values) # using the continuous predictions from SVM, extracting as 1D array

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Computing Covariance for Predicates
Metric: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_3 | Sample

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 2.70
1,Ag La <= 7.76,Ag La <= 7.76,Ag La <= 7.76,Ag La <= 7.76
2,Ag Lb <= 2.36,Ag Lb <= 2.36,Ag Lb <= 2.36,Ag Lb <= 2.36
3,Ag Lb <= 6.48,Ag Lb <= 6.48,Ag Lb <= 6.48,Ag Lb <= 6.48
4,Ag La <= 9.01,Ag La <= 9.01,Ag La <= 9.01,Ag La <= 9.01
...,...,...,...,...
57,Cu <= -0.64,Fe <= 0.31,background > 0.40,Cu > -0.64
58,background > -0.46,Ag La > 7.76,background <= 1.29,background <= 0.40
59,Fe <= -0.36,background > 0.40,Fe <= -0.36,background <= -0.46
60,Class_A,Class_A,Class_A,Class_A


In [8]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 2.70,28.218676,Ag La,2.70,<=
1,Ag Lb <= 2.36,20.990665,Ag Lb,2.36,<=
2,Ag ka <= 5.37,0.817253,Ag ka,5.37,<=
3,Bremsstrahlung <= 4.15,0.720486,Bremsstrahlung,4.15,<=
4,Ag compton <= 6.52,0.605184,Ag compton,6.52,<=
5,Cu > 0.32,0.289875,Cu,0.32,>
6,Ca <= -0.68,0.279436,Ca,-0.68,<=
7,background <= -0.46,0.251596,background,-0.46,<=
8,Fe > -1.18,0.239142,Fe,-1.18,>
9,Zn > 0.56,0.228150,Zn,0.56,>


In [9]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zone 'Ag La': VE = 91.39%, dim = 18 variables
Zone 'Ag Lb': VE = 90.27%, dim = 15 variables
Zone 'Ca': VE = 27.70%, dim = 19 variables
Zone 'background': VE = 3.72%, dim = 87 variables
Zone 'Fe': VE = 14.68%, dim = 23 variables
Zone 'Cu': VE = 15.48%, dim = 66 variables
Zone 'Zn': VE = 21.60%, dim = 29 variables
Zone 'Bremsstrahlung': VE = 38.87%, dim = 429 variables
Zone 'Ag compton': VE = 52.69%, dim = 62 variables
Zone 'Ag ka': VE = 80.83%, dim = 45 variables


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ag La <= 2.70,28.218676,Ag La,2.70,<=,34.935039,86.0,0.021418,Ag La <= 34.935039
1,Ag La <= 7.76,24.327379,Ag La,7.76,<=,105.814886,78.0,0.002963,Ag La <= 105.814886
2,Ag Lb <= 2.36,20.990665,Ag Lb,2.36,<=,30.578162,0.0,0.008013,Ag Lb <= 30.578162
3,Ag Lb <= 6.48,16.657994,Ag Lb,6.48,<=,75.496450,98.0,0.013050,Ag Lb <= 75.496450
4,Ag La <= 9.01,10.865589,Ag La,9.01,<=,126.903607,239.0,0.004801,Ag La <= 126.903607
...,...,...,...,...,...,...,...,...,...
57,Cu > -0.64,0.134470,Cu,-0.64,>,-1.435498,17.0,0.000042,Cu > -1.435498
58,background > 0.40,0.128819,background,0.40,>,4.852293,186.0,0.003287,background > 4.852293
59,Fe <= -0.36,0.104788,Fe,-0.36,<=,0.719116,9.0,0.001198,Fe <= 0.719116
60,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [10]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ag La':
  - Dimensão: 18 variáveis espectrais
  - Range de energias: 2.66 - 3.10
  - Variância explicada pela PC1: 91.39%


In [11]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=mlp_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 60 | Discarded: 20
PERTURBATION IMPORTANCE FOR PREDICATES
Task type (aim): classification
Perturbation mode: median
Statistics source: full
Metric: probability_shift
Total folds: 10


[Bag_1] Processing 60 predicates...
  Predicate: Ag La > -18.52 (n=176)
    Zone: 18 colu

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 2.70
1,Bremsstrahlung <= -1.54,Bremsstrahlung <= -1.54,Bremsstrahlung <= -1.54,Ag La <= 7.76
2,Ag Lb <= 2.36,Ag Lb <= 2.36,Ag Lb <= 2.36,Bremsstrahlung <= -1.54
3,Bremsstrahlung <= 4.15,Ag La <= 7.76,Bremsstrahlung <= 4.15,Bremsstrahlung <= 4.15
4,Ag La <= 9.01,Ag La <= 9.01,Ag La <= 7.76,Ag Lb <= 2.36
...,...,...,...,...
57,Ca <= -0.68,Ca <= -0.68,Ca > 0.46,Ca > -1.79
58,Ca > -1.79,Ca <= 1.81,Ca > -1.79,Ca <= 0.46
59,Ca <= 1.81,Ca > -0.68,Ca <= 1.81,Ca <= 1.81
60,Class_A,Class_A,Class_A,Class_A


In [12]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ag La <= 2.70,0.218193
1,Bremsstrahlung <= -1.54,0.183728
2,Ag Lb <= 2.36,0.165061
3,Bremsstrahlung <= 4.15,0.151607
4,Bremsstrahlung <= 9.64,0.145160
5,Ag La <= 7.76,0.141935
6,Bremsstrahlung > -9.06,0.119050
7,Bremsstrahlung > 4.15,0.114600
8,Ag La <= 9.01,0.113973
9,Ag Lb <= 6.48,0.109012


In [13]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ag La <= 2.70,6.043251,Ag La,2.70,<=
1,Bremsstrahlung <= -1.54,3.741575,Bremsstrahlung,-1.54,<=
2,Ag Lb <= 2.36,3.522829,Ag Lb,2.36,<=
3,Ag ka > 5.37,2.365298,Ag ka,5.37,>
4,Ag compton <= -1.09,0.553964,Ag compton,-1.09,<=
5,background <= 1.29,0.238849,background,1.29,<=
6,Zn <= 0.56,0.121458,Zn,0.56,<=
7,Cu > -1.74,0.107682,Cu,-1.74,>
8,Fe <= -0.36,0.021631,Fe,-0.36,<=
9,Ca > 0.46,0.013500,Ca,0.46,>


In [14]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ag La <= 2.70,6.043251,Ag La,2.70,<=,34.935039,86.0,0.021418,Ag La <= 34.935039
1,Bremsstrahlung <= -1.54,3.741575,Bremsstrahlung,-1.54,<=,-72.079168,60.0,0.037184,Bremsstrahlung <= -72.079168
2,Ag Lb <= 2.36,3.522829,Ag Lb,2.36,<=,30.578162,0.0,0.008013,Ag Lb <= 30.578162
3,Ag La <= 7.76,3.153507,Ag La,7.76,<=,105.814886,78.0,0.002963,Ag La <= 105.814886
4,Bremsstrahlung <= 4.15,3.063384,Bremsstrahlung,4.15,<=,83.677742,106.0,0.010264,Bremsstrahlung <= 83.677742
...,...,...,...,...,...,...,...,...,...
57,Ca > -1.79,0.009745,Ca,-1.79,>,-9.483856,153.0,0.012409,Ca > -9.483856
58,Ca > -0.68,0.008865,Ca,-0.68,>,-8.742470,9.0,0.004755,Ca > -8.742470
59,Ca <= 1.81,0.007141,Ca,1.81,<=,-2.575534,222.0,0.044016,Ca <= -2.575534
60,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


In [15]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ag La':
  - Dimensão: 18 variáveis espectrais
  - Variância explicada pela PC1: 91.39%


In [16]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = mlp_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = mlp_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,22.31,0.032329,Ag ka
1,3.02,0.022214,Ag La
2,3.17,0.021363,Ag Lb
3,20.70,0.011740,Ag compton
4,8.73,0.010901,Zn
5,19.50,0.008636,Bremsstrahlung
6,6.86,0.006604,Cu
7,5.89,0.006122,background
8,6.40,0.005054,Fe
9,3.74,0.004203,Ca


In [17]:
import numpy as np

max_len = max(
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Ag Lb,Ag ka,Ag La,Ag La
1,Ag La,Ag La,Bremsstrahlung,Ag Lb
2,Ag ka,Ag Lb,Ag Lb,Ag ka
3,Zn,Ag compton,Ag ka,Bremsstrahlung
4,Ag compton,Zn,Ag compton,Ag compton
5,Bremsstrahlung,Bremsstrahlung,background,Cu
6,background,Cu,Zn,Ca
7,Cu,background,Cu,background
8,Fe,Fe,Fe,Fe
9,Ca,Ca,Ca,Zn


In [18]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_16960\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
5,LRC_perturbation,LRC_covariance,0.794255
2,Shap,LRC_covariance,0.608124
0,Shap,Permutation,0.535985
4,Permutation,LRC_covariance,0.508166
1,Shap,LRC_perturbation,0.469218
3,Permutation,LRC_perturbation,0.464176


In [19]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')